In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.gridspec as gridspec

# Create a sphere mesh
phi = np.linspace(0, np.pi, 100)  # Polar angle (latitude)
theta = np.linspace(0, 2 * np.pi, 100) # Azimuthal angle (longitude)
phi, theta = np.meshgrid(phi, theta)

x = np.sin(phi) * np.cos(theta)
y = np.sin(phi) * np.sin(theta)
z = np.cos(phi)

plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

def plot_spherical_harmonic(ax, l, m_signed, x, y, z, title, show_axes=False):
    # SciPy's sph_harm_y expects (n, m, polar_angle, azimuthal_angle)
    # In this script, 'phi' is polar, 'theta' is azimuthal.
    Ylm_complex = sph_harm_y(l, m_signed, phi, theta)
    Ylm = Ylm_complex.real

    # Normalize Ylm values for colormapping to [-1, 1]
    max_abs_Ylm = np.max(np.abs(Ylm))
    if max_abs_Ylm == 0:
        normalized_Ylm = np.zeros_like(Ylm)
    else:
        normalized_Ylm = Ylm / max_abs_Ylm

    # Plot the surface
    # Use cmap='RdBu_r' for red=positive, blue=negative (RdBu is blue=positive, red=negative)
    # Apply shading only for l=0 to make it look more spherical
    if l == 0:
        surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm * 0.4 + 0.5), shade=True, antialiased=True)
    else:
        surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm * 0.5 + 0.5), shade=False, antialiased=True)

    # Move title to underneath the plot and adjust y position
    ax.text2D(0.5, -0.1, title, transform=ax.transAxes, ha='center', va='top', fontsize=30)

    if show_axes:
        ax.set_axis_on()
        ax.set_xlabel(r'$x$', fontsize=14, labelpad=5)
        ax.set_ylabel(r'$y$', fontsize=14, labelpad=5)
        ax.set_zlabel(r'$z$', fontsize=14, labelpad=5)
        # Reduce the number of ticks on the axes, with labels every 0.5
        ax.set_xticks(np.arange(-1, 1.1, 0.5))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_zticks(np.arange(-1, 1.1, 0.5))
    else:
        ax.set_axis_off()

    ax.view_init(elev=20, azim=20) # Set a consistent viewpoint for all subplots

# Create the plot
fig = plt.figure(figsize=(20, 16), constrained_layout=True)

gs = fig.add_gridspec(nrows=3, ncols=5) # 3 rows for l=0,1,2 and 5 max columns for l=2

row_idx = 0

# l = 0
l_val = 0
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 1 plot in 5 columns, start at (5-1)/2 = 2
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Real)', show_axes=True)
row_idx += 1

# l = 1
l_val = 1
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 3 plots in 5 columns, start at (5-3)/2 = 1
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Real)', show_axes=True)
row_idx += 1

# l = 2
l_val = 2
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 5 plots in 5 columns, start at (5-5)/2 = 0
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Real)', show_axes=True)
row_idx += 1

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.gridspec as gridspec


plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

def plot_imaginary_colored_spherical_harmonic(ax, l, m_signed, x_unit, y_unit, z_unit, title, show_axes=False):
    # For m_signed = 0, the imaginary part of spherical harmonics is always zero.
    # To make the plot truly empty as requested, we return early.
    if m_signed == 0:
        ax.set_axis_off() # Ensure axes are off for empty plots
        ax.set_title(title, y=-0.1, ha='center', va='top', fontsize=30) # Still show title
        ax.set_xlim([-1, 1]) # Keep consistent limits even if empty
        ax.set_ylim([-1, 1])
        ax.set_zlim([-1, 1])
        ax.view_init(elev=20, azim=20)
        return

    # SciPy's sph_harm_y expects (n, m, polar_angle, azimuthal_angle)
    # In this script, 'phi' is polar, 'theta' is azimuthal.
    Ylm_complex = sph_harm_y(l, m_signed, phi, theta)
    Ylm = Ylm_complex.imag # Use the imaginary part for coloring

    # Normalize Ylm values for colormapping to [-1, 1]
    max_abs_Ylm = np.max(np.abs(Ylm))
    if max_abs_Ylm == 0:
        normalized_Ylm = np.zeros_like(Ylm)
    else:
        normalized_Ylm = Ylm / max_abs_Ylm

    # Plot the surface (on the unit sphere, no deformation)
    # Use cmap='RdBu_r' for red=positive, blue=negative
    if l == 0:
        surf = ax.plot_surface(x_unit, y_unit, z_unit, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm * 0.4 + 0.5), shade=True, antialiased=True)
    else:
        surf = ax.plot_surface(x_unit, y_unit, z_unit, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm * 0.5 + 0.5), shade=False, antialiased=True)

    # Move title to underneath the plot and adjust y position
    ax.text2D(0.5, -0.1, title, transform=ax.transAxes, ha='center', va='top', fontsize=30)

    if show_axes:
        ax.set_axis_on()
        ax.set_xlabel(r'$x$', fontsize=14, labelpad=5)
        ax.set_ylabel(r'$y$', fontsize=14, labelpad=5)
        ax.set_zlabel(r'$z$', fontsize=14, labelpad=5)
        # Reduce the number of ticks on the axes, with labels every 0.5
        ax.set_xticks(np.arange(-1, 1.1, 0.5))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_zticks(np.arange(-1, 1.1, 0.5))
    else:
        ax.set_axis_off()

    ax.view_init(elev=20, azim=20) # Set a consistent viewpoint for all subplots

# Create the plot figure
fig = plt.figure(figsize=(20, 16), constrained_layout=True)
gs = fig.add_gridspec(nrows=3, ncols=5) # 3 rows for l=0,1,2 and 5 max columns for l=2

row_idx = 0

# l = 0
l_val = 0
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 1 plot in 5 columns, start at (5-1)/2 = 2
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    # Note: 'x', 'y', 'z' are assumed to be defined from previous cells as unit sphere coordinates.
    plot_imaginary_colored_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Imaginary)', show_axes=True)
row_idx += 1

# l = 1
l_val = 1
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 3 plots in 5 columns, start at (5-3)/2 = 1
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_imaginary_colored_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Imaginary)', show_axes=True)
row_idx += 1

# l = 2
l_val = 2
num_m = 2 * l_val + 1
m_start = -(l_val)
# To center 5 plots in 5 columns, start at (5-5)/2 = 0
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_imaginary_colored_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$' + '\n(Imaginary)', show_axes=True)
row_idx += 1

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y

# Reuse the coordinate mesh from the first cell
# (If not already run, we redefine it here for completeness)
phi_vals = np.linspace(0, np.pi, 100)  # Polar
theta_vals = np.linspace(0, 2 * np.pi, 100) # Azimuthal
phi_mesh, theta_mesh = np.meshgrid(phi_vals, theta_vals)

x_sphere = np.sin(phi_mesh) * np.cos(theta_mesh)
y_sphere = np.sin(phi_mesh) * np.sin(theta_mesh)
z_sphere = np.cos(phi_mesh)

def plot_probability_density_sphere(ax, l, m, x, y, z, title, show_axes=False):
    # Calculate complex spherical harmonic
    Ylm = sph_harm_y(l, m, phi_mesh, theta_mesh)

    # Calculate probability density |Ylm|^2
    prob = np.abs(Ylm)**2

    # Normalize for colormapping [0, 1]
    norm_prob = prob / np.max(prob) if np.max(prob) > 0 else prob

    # Plot on the unit sphere surface
    # 'winter' colormap leans towards blue/cyan
    ax.plot_surface(x, y, z, rstride=1, cstride=1,
                    facecolors=plt.cm.GnBu(norm_prob),
                    shade=True, antialiased=True)

    # Styling
    ax.set_title(title, y=-0.15, fontsize=30)

    if show_axes:
        ax.set_axis_on()
        ax.set_xlabel(r'$x$', fontsize=12, labelpad=0)
        ax.set_ylabel(r'$y$', fontsize=12, labelpad=0)
        ax.set_zlabel(r'$z$', fontsize=12, labelpad=0)
        # Set ticks every 0.5
        ax.set_xticks(np.arange(-1, 1.1, 0.5))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_zticks(np.arange(-1, 1.1, 0.5))
        # Make tick labels smaller for clarity in the grid
        ax.tick_params(axis='both', which='major', labelsize=8)
    else:
        ax.set_axis_off()

    ax.view_init(elev=20, azim=30)

# Create the grid for l=0, 1, 2
fig = plt.figure(figsize=(18, 12), constrained_layout=True)
gs = fig.add_gridspec(3, 5)

for l in range(3):
    m_range = range(-l, l + 1)
    start_col = (5 - len(m_range)) // 2
    for i, m in enumerate(m_range):
        ax = fig.add_subplot(gs[l, start_col + i], projection='3d')
        plot_probability_density_sphere(ax, l, m, x_sphere, y_sphere, z_sphere,
                                         rf'$|Y_{{{l}}}^{{{m}}}|^2$', show_axes=True)

#plt.suptitle(r'Spherical Harmonic Probability Density $|Y_ℓ^m|^2$ on a Unit Sphere', fontsize=22)
plt.show()

In [ ]:
def plot_real_deformed_spherical_harmonic(ax, l, m_signed, x_unit, y_unit, z_unit, title, show_axes=False):
    # SciPy's sph_harm_y expects (n, m, polar_angle, azimuthal_angle)
    # In this script, 'phi' is polar, 'theta' is azimuthal.
    Ylm_complex = sph_harm_y(l, m_signed, phi, theta)

    axislim=1.0

    # Calculate real part for both radial distance (absolute value) and coloring
    Ylm_real_part = Ylm_complex.real

    # Deform the unit sphere based on the absolute value of the real part
    R = np.abs(Ylm_real_part)

    # Scale R to fill the axis limits more effectively if it's not already large enough
    max_R_value = np.max(R)
    if max_R_value > 0:
        # Ensure the largest extent of the deformed shape reaches close to axislim
        # Using 0.9 to leave a small margin, can be adjusted
        R = R / max_R_value * axislim * 0.9

    # Normalize Ylm_real_part values for colormapping to [-1, 1]
    max_abs_Ylm_real_part = np.max(np.abs(Ylm_real_part))
    if max_abs_Ylm_real_part == 0:
        normalized_Ylm_real_part = np.zeros_like(Ylm_real_part)
    else:
        normalized_Ylm_real_part = Ylm_real_part / max_abs_Ylm_real_part

    # Deform the unit sphere coordinates using R as radial distance
    x = x_unit * R
    y = y_unit * R
    z = z_unit * R

    # Plot the deformed surface, colored by the actual real part's sign
    # Use cmap='RdBu_r' for red=positive, blue=negative
    if l == 0:
        surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm_real_part * 0.4 + 0.5), shade=True, antialiased=True, zorder=1)
    else:
        surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm_real_part * 0.5 + 0.5), shade=False, antialiased=True, zorder=1)

    # Move title to underneath the plot and adjust y position for two lines
    ax.text2D(0.5, -0.1, f"{title}\n(Real Deformed)", transform=ax.transAxes, ha='center', va='top', fontsize=30)


    if show_axes:
        ax.set_axis_on()
        ax.set_xlabel('X', fontsize=14, labelpad=5)
        ax.set_ylabel('Y', fontsize=14, labelpad=5)
        ax.set_zlabel('Z', fontsize=14, labelpad=5)
        # Reduce the number of ticks on the axes, with labels every 0.5
        ax.set_xticks(np.arange(-1, 1.1, 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.25))
        ax.set_zticks(np.arange(-1, 1.1, 0.25))
        # Set axis limits to -0.5 to 0.5
        ax.set_xlim([-axislim, axislim])
        ax.set_ylim([-axislim, axislim])
        ax.set_zlim([-axislim, axislim])
    else:
        ax.set_axis_off()
        # Explicitly set limits when axes are off to prevent distortion from auto-scaling with nodal planes
        ax.set_xlim([-axislim, axislim])
        ax.set_ylim([-axislim, axislim])
        ax.set_zlim([-axislim, axislim])

    ax.view_init(elev=20, azim=20) # Set a consistent viewpoint for all subplots

    # Call plot_nodal_planes to add nodal planes
    plot_nodal_planes(ax, l, m_signed, 'real', show_axes=show_axes)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.gridspec as gridspec

def plot_nodal_planes(ax, l, m_signed, component_type, show_axes=False):
    # Ensure we use the current axis limits for sizing
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    zmin, zmax = ax.get_zlim()

    plot_max_extent = max(abs(xmin), abs(xmax), abs(ymin), abs(ymax), abs(zmin), abs(zmax))

    # Increased alpha for better visibility and added edgecolors
    general_alpha = 0.5
    line_alpha = 0.6

    # Plot z=0 nodal plane (Equatorial Node)
    # Occurs if l - |m| is odd (e.g., l=1, m=0 or l=2, m=1)
    if (l - abs(m_signed)) % 2 != 0:
        X_plane = np.linspace(xmin, xmax, 2)
        Y_plane = np.linspace(ymin, ymax, 2)
        X_plane, Y_plane = np.meshgrid(X_plane, Y_plane)
        Z_plane = np.zeros_like(X_plane)
        ax.plot_surface(X_plane, Y_plane, Z_plane, color='purple', alpha=general_alpha,
                        edgecolor='indigo', lw=0.5, antialiased=True, zorder=5)

    # Plot vertical nodal planes
    if m_signed != 0:
        theta_angles = []
        if component_type == 'real':
            for k in range(abs(m_signed)):
                theta_angles.append((k + 0.5) * np.pi / abs(m_signed))
        elif component_type == 'imaginary':
            for k in range(abs(m_signed)):
                theta_angles.append(k * np.pi / abs(m_signed))

        for theta_plane in theta_angles:
            r_plane_vals = np.linspace(-plot_max_extent, plot_max_extent, 2)
            z_plane_vals = np.linspace(zmin, zmax, 2)
            R_mesh, Z_mesh = np.meshgrid(r_plane_vals, z_plane_vals)
            X_plane = R_mesh * np.cos(theta_plane)
            Y_plane = R_mesh * np.sin(theta_plane)
            ax.plot_surface(X_plane, Y_plane, Z_mesh, color='purple', alpha=general_alpha,
                            edgecolor='indigo', lw=0.5, antialiased=True, zorder=5)

    # Plot conical nodal surfaces for m_signed = 0, l > 1 (e.g. l=2, m=0)
    if m_signed == 0 and l > 1:
        conical_alpha = 0.5
        max_r_cone = plot_max_extent * 1.1

        # Roots of P_l(cos(phi)) = 0
        roots = np.polynomial.legendre.Legendre.basis(l).roots()
        valid_roots = roots[(roots > -1) & (roots < 1)]
        phi_nodes = np.arccos(valid_roots)

        for phi_node in phi_nodes:
            if np.isclose(phi_node, np.pi/2): continue # Handled by equatorial check

            r_vals = np.linspace(0, max_r_cone, 30)
            theta_vals = np.linspace(0, 2 * np.pi, 60)
            R_mesh, T_mesh = np.meshgrid(r_vals, theta_vals)

            X_cone = R_mesh * np.sin(phi_node) * np.cos(T_mesh)
            Y_cone = R_mesh * np.sin(phi_node) * np.sin(T_mesh)
            Z_cone = R_mesh * np.cos(phi_node)

            ax.plot_surface(X_cone, Y_cone, Z_cone, color='purple', alpha=conical_alpha,
                            edgecolor='indigo', lw=0.2, antialiased=True, zorder=5)

In [ ]:
fig = plt.figure(figsize=(20, 16), constrained_layout=True)
gs = fig.add_gridspec(nrows=3, ncols=5)

# Refresh variables from initial mesh
phi_vals = np.linspace(0, np.pi, 100)
theta_vals = np.linspace(0, 2 * np.pi, 100)
phi, theta = np.meshgrid(phi_vals, theta_vals)
x_sphere = np.sin(phi) * np.cos(theta)
y_sphere = np.sin(phi) * np.sin(theta)
z_sphere = np.cos(phi)

row_idx = 0
for l_val in [0, 1, 2]:
    num_m = 2 * l_val + 1
    m_start = -l_val
    start_col = (5 - num_m) // 2
    for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
        ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
        # Use explicit sphere variables to ensure consistency
        plot_real_deformed_spherical_harmonic(ax, l_val, m_signed, x_sphere, y_sphere, z_sphere,
                                              rf'$\ell={l_val}$, $m_\ell={m_signed}$', show_axes=False)
    row_idx += 1

plt.show()

In [ ]:
def plot_imaginary_spherical_harmonic(ax, l, m_signed, x_unit, y_unit, z_unit, title, show_axes=False):
    # SciPy's sph_harm_y expects (n, m, polar_angle, azimuthal_angle)
    # In this script, 'phi' is polar, 'theta' is azimuthal.
    Ylm_complex = sph_harm_y(l, m_signed, phi, theta)

    axislim=1.0

    # Calculate absolute value for radial distance
    R = np.abs(Ylm_complex)

    # Scale R to fill the axis limits more effectively if it's not already large enough
    max_R_value = np.max(R)
    if max_R_value > 0:
        # Ensure the largest extent of the deformed shape reaches close to axislim
        # Using 0.9 to leave a small margin, can be adjusted
        R = R / max_R_value * axislim * 0.9

    # Calculate imaginary part for coloring
    Ylm_imag = Ylm_complex.imag

    # Normalize Ylm_imag values for colormapping to [-1, 1]
    max_abs_Ylm_imag = np.max(np.abs(Ylm_imag))
    if max_abs_Ylm_imag == 0:
        normalized_Ylm_imag = np.zeros_like(Ylm_imag)
    else:
        normalized_Ylm_imag = Ylm_imag / max_abs_Ylm_imag

    # Deform the unit sphere coordinates using R as radial distance
    x = x_unit * R
    y = y_unit * R
    z = z_unit * R

    # Plot the deformed surface
    # Use cmap='RdBu_r' for red=positive, blue=negative
    if m_signed == 0:
        surf = 0
    else:
        surf = ax.plot_surface(x, y, z, rstride=1, cstride=1, facecolors=plt.cm.RdBu_r(normalized_Ylm_imag * 0.5 + 0.5), shade=False, antialiased=True, zorder=1)

    # Move title to underneath the plot and adjust y position for two lines
    ax.text2D(0.5, -0.10, f"{title}\n(Imaginary\nDeformed)", transform=ax.transAxes, ha='center', va='top', fontsize=30)

    if show_axes:
        ax.set_axis_on()
        ax.set_xlabel('X', fontsize=14, labelpad=5)
        ax.set_ylabel('Y', fontsize=14, labelpad=5)
        ax.set_zlabel('Z', fontsize=14, labelpad=5)
        # Reduce the number of ticks on the axes, with labels every 0.5
        ax.set_xticks(np.arange(-1, 1.1, 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.25))
        ax.set_zticks(np.arange(-1, 1.1, 0.25))
        # Set axis limits to -0.5 to 0.5
        ax.set_xlim([-axislim, axislim])
        ax.set_ylim([-axislim, axislim])
        ax.set_zlim([-axislim, axislim])
    else:
        ax.set_axis_off()
        # Explicitly set limits when axes are off to prevent distortion from auto-scaling with nodal planes
        ax.set_xlim([-axislim, axislim])
        ax.set_ylim([-axislim, axislim])
        ax.set_zlim([-axislim, axislim])

    ax.view_init(elev=20, azim=20) # Set a consistent viewpoint for all subplots

    # Call plot_nodal_planes to add nodal planes
    if m_signed != 0:
        plot_nodal_planes(ax, l, m_signed, 'imaginary', show_axes=show_axes)

In [ ]:
fig = plt.figure(figsize=(20, 16), constrained_layout=True)
gs = fig.add_gridspec(nrows=3, ncols=5)

# Variables x, y, z are already defined as unit sphere coordinates from the initial setup.

plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm' # Use Computer Modern for math text, which is serif

row_idx = 0

# l = 0
l_val = 0
num_m = 2 * l_val + 1
m_start = -(l_val)
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_imaginary_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$', show_axes=False)
row_idx += 1

# l = 1
l_val = 1
num_m = 2 * l_val + 1
m_start = -(l_val)
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_imaginary_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$', show_axes=False)
row_idx += 1

# l = 2
l_val = 2
num_m = 2 * l_val + 1
m_start = -(l_val)
start_col = (5 - num_m) // 2
for m_offset, m_signed in enumerate(range(m_start, l_val + 1)):
    ax = fig.add_subplot(gs[row_idx, start_col + m_offset], projection='3d')
    plot_imaginary_spherical_harmonic(ax, l_val, m_signed, x, y, z, rf'$\ell={l_val}$, $m_\ell={m_signed}$', show_axes=False)
row_idx += 1

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set font styles consistent with previous plots
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

# Define the quantum numbers l
l_max = 4
l_values = np.arange(0, l_max + 1)
# Energy in units of hbar^2 / 2I
energies = l_values * (l_values + 1)

# Widened the figure further
fig, ax = plt.subplots(figsize=(16, 12))

# Define updated geometry for the state lines
segment_width = 0.09
segment_gap = 0.01
center_x = 0.5

for l, e in zip(l_values, energies):
    # Number of degenerate states (2l + 1)
    num_states = 2 * l + 1

    # Calculate the total width required to center the states
    total_width = num_states * segment_width + (num_states - 1) * segment_gap
    start_x = center_x - total_width / 2

    for i in range(num_states):
        x_s = start_x + i * (segment_width + segment_gap)
        x_e = x_s + segment_width

        # Lines are blue with linewidth 4 as in the last successful state
        ax.hlines(e, x_s, x_e, colors='blue', linewidth=4)

        # Label the full m_l expression below the segments
        m_l = i - l
        ax.text((x_s + x_e)/2, e - 0.2, rf'$m_\ell={m_l}$', ha='center', va='top', fontsize=18, color='blue', alpha=0.8)

    # Label the quantum number l
    ax.text(start_x + total_width + 0.02, e, rf'$\ell={l}$', va='center', fontsize=20)

# Set y-axis ticks at specific energy levels with fraction formatting
ax.set_yticks(energies)
y_tick_labels = [rf'${int(e)} \frac{{\hbar^2}}{{2I}}$' if e != 0 else "0" for e in energies]
ax.set_yticklabels(y_tick_labels, fontsize=20)

ax.set_ylabel(r'Energy', fontsize=22)
# Expanded x-axis limits to accommodate the wide l=5 level and labels
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-3, l_max*(l_max+1) + 3)
ax.set_xticks([]) # Remove x-axis ticks

# Add a grid for the energy axis
ax.yaxis.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()